# End to end distributed training on a Databricks Notebook

Distributed training on PyTorch is often done by creating a file (`train.py`) and using the `torchrun` CLI to run distributed training using that file. Databricks offers a method of doing distributed training directly on a Databricks notebook. You can define the `train()` function within a notebook and use the `TorchDistributor` API to train the model across the workers.

This notebook illustrates how to develop interactively within a notebook. Particularly with larger deep learning projects, Databricks recommends leveraging the `%run` command in order to split up your code into manageable chunks.

In this notebook, you: 
- Train a simple single GPU model on the classic MNIST dataset 
- Adapt that code for distributed training 
- Learn how the TorchDistributor can be leveraged to help you scale up the model training across multiple GPUs or multiple nodes. 

## Requirements
- Databricks Runtime ML 13.0 and above
- This notebook should be run on a cluster with Single User access mode. If the cluster should be shared with other team members, contact your Databricks account team for solutions.
- (Recommended) GPU instances [AWS](https://docs.databricks.com/clusters/gpu.html) | [Azure](https://learn.microsoft.com/en-gb/azure/databricks/clusters/gpu) | [GCP](https://docs.gcp.databricks.com/clusters/gpu.html)

## Tested on
- Single-node:
  - Standard_NC12s_v3 (2 x V100, 224 GB Memory, 12 Cores)
- Multi-node:
  - Driver: Standard_DS3_v2 (14 GB, 4 Cores)
  - Worker: Standard_NC12s_v3 (2 x V100, 224 GB Memory, 12 Cores)
- DBR 16.4 LTS ML
- Dedicated access mode


### MLflow setup

MLflow is a tool to support the tracking of machine learning experiments and logging of models. The `db_host` variable controls the MLflow tracking server and needs to be set to the URL of the workspace.

***NOTE*** The MLflow PyTorch Autologging APIs are designed for PyTorch Lightning and won't work with Native PyTorch

In [0]:
%sh ls /Workspace/Users/victor.rodrigues@databricks.com/databricks-demos/DL

In [0]:
import mlflow

experiment_path = f'/Workspace/Users/victor.rodrigues@databricks.com/torch-distributor'

# Add your workspace URL and API token here 
db_host = 'https://e2-demo-field-eng.cloud.databricks.com/'
db_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Manually create the experiment so that you know the ID and can send that to the worker nodes when you are ready to scale
experiment = mlflow.set_experiment(experiment_path)

## Define train and test functions

The following cell contains code that describes the model, the train function, and the testing function; all of which are designed to run locally. Next, the code introduces the changes needed to move training from the local setting to a distributed setting.

All the torch code leverages standard PyTorch APIs, there are no custom libraries required or alterations in the way the code is written. This notebook focuses on how to scale your training with `TorchDistributor` and does not go through the model code. 

In [0]:
import torch
NUM_WORKERS = 2
NUM_GPUS_PER_NODE = torch.cuda.device_count()

In [0]:
%sql create volume vr_demo.dl.serverless_gpu

In [0]:
PYTORCH_DIR = '/Volumes/vr_demo/dl/serverless_gpu'

batch_size = 100
num_epochs = 3
momentum = 0.5
log_interval = 100
learning_rate = 0.001

import torch
import torch.nn as nn
import torch.nn.functional as F

# Our Model definition
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x)

def train_one_epoch(model, device, data_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(data_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(data_loader) * len(data),
                100. * batch_idx / len(data_loader), loss.item()))
            
            mlflow.log_metric('train_loss', loss.item())

def save_checkpoint(log_dir, model, optimizer, epoch):
  filepath = log_dir + '/checkpoint-{epoch}.pth.tar'.format(epoch=epoch)
  state = {
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
  }
  torch.save(state, filepath)
  
def load_checkpoint(log_dir, epoch=num_epochs):
  filepath = log_dir + '/checkpoint-{epoch}.pth.tar'.format(epoch=epoch)
  return torch.load(filepath)

def create_log_dir():
  log_dir = os.path.join(PYTORCH_DIR, str(time()))
  os.makedirs(log_dir)
  return log_dir

import torch.optim as optim
from torchvision import datasets, transforms
from time import time
import os

base_log_dir = create_log_dir()
print("Log directory:", base_log_dir)

# Import the distributed decorator
from serverless_gpu import distributed

# Decorate the function with @distributed and specify the number of GPUs, the GPU type, and whether or not the GPUs are remote
@distributed(gpus=2, gpu_type='A10', remote=True)
def train(log_dir):
  device = torch.device('cuda')

  train_parameters = {'batch_size': batch_size, 'epochs': num_epochs}
  mlflow.log_params(train_parameters)
  
  train_dataset = datasets.MNIST(
    'data', 
    train=True,
    download=True,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]))
  data_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

  model = Net().to(device)

  optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum)

  for epoch in range(1, num_epochs + 1):
    train_one_epoch(model, device, data_loader, optimizer, epoch)
    save_checkpoint(log_dir, model, optimizer, epoch)
    
def test(log_dir):
  device = torch.device('cuda')
  loaded_model = Net().to(device)  

  checkpoint = load_checkpoint(log_dir)
  loaded_model.load_state_dict(checkpoint['model'])
  loaded_model.eval()

  test_dataset = datasets.MNIST(
    'data', 
    train=False,
    download=True,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]))
  data_loader = torch.utils.data.DataLoader(test_dataset)

  test_loss = 0
  for data, target in data_loader:
      data, target = data.to(device), target.to(device)
      output = loaded_model(data)
      test_loss += F.nll_loss(output, target)
        
  test_loss /= len(data_loader.dataset)
  print("Average test loss: {}".format(test_loss.item()))
  
  mlflow.log_metric('test_loss', test_loss.item())
  
  mlflow.pytorch.log_model(loaded_model, "model")

### Train the model locally

To test that this runs correctly, you can trigger a train and test iteration using the functions defined above.

In [0]:
with mlflow.start_run(run_name='single_node'):
  mlflow.log_param('run_type', 'local')
  train.distributed(base_log_dir)
  # test(base_log_dir)

## Distributed Setup

When you wrap the single-node code in the `train()` function, Databricks recommends you include all the import statements inside the `train()` function to avoid library pickling issues.

Everything else is what is normally required for getting distributed training to work within PyTorch.
- Calling `dist.init_process_group("nccl")` at the beginning of `train()`
- Calling `dist.destroy_process_group()` at the end of `train()`
- Setting `local_rank = int(os.environ["LOCAL_RANK"])`
- Adding a `DistributedSampler` to the `DataLoader`
- Wrapping the model with a `DDP(model)`
- For more information, view https://pytorch.org/tutorials/intermediate/ddp_series_multinode.html

In [0]:
single_node_single_gpu_dir = create_log_dir()
print("Data is located at: ", single_node_single_gpu_dir)

def train_one_epoch(model, device, data_loader, optimizer, epoch):
  model.train()
  for batch_idx, (data, target) in enumerate(data_loader):
    data, target = data.to(device), target.to(device)
    optimizer.zero_grad()
    output = model(data)
    loss = F.nll_loss(output, target)
    loss.backward()
    optimizer.step()
    if batch_idx % log_interval == 0:
      print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
          epoch, batch_idx * len(data), len(data_loader) * len(data),
          100. * batch_idx / len(data_loader), loss.item()))
      
      if int(os.environ["RANK"]) == 0:
        mlflow.log_metric('train_loss', loss.item())

def save_checkpoint(log_dir, model, optimizer, epoch):
  filepath = log_dir + '/checkpoint-{epoch}.pth.tar'.format(epoch=epoch)
  state = {
    'model': model.module.state_dict(),
    'optimizer': optimizer.state_dict(),
  }
  torch.save(state, filepath)

# For distributed training we will merge the train and test steps into 1 main function
def main_fn(directory):
  
  #### Added imports here ####
  import mlflow
  import torch.distributed as dist
  from torch.nn.parallel import DistributedDataParallel as DDP
  from torch.utils.data.distributed import DistributedSampler
  
  ############################

  ##### Setting up MLflow ####
  # We need to do this so that different processes that will be able to find mlflow
  os.environ['DATABRICKS_HOST'] = db_host
  os.environ['DATABRICKS_TOKEN'] = db_token

  # We set the experiment details here
  experiment = mlflow.set_experiment(experiment_path)
  ############################
  
  print("Running distributed training")
  dist.init_process_group("nccl")
  
  local_rank = int(os.environ["LOCAL_RANK"])
  global_rank = int(os.environ["RANK"])
  
  if global_rank == 0:
    train_parameters = {'batch_size': batch_size, 'epochs': num_epochs, 'trainer': 'TorchDistributor'}
    mlflow.log_params(train_parameters)
  
  train_dataset = datasets.MNIST(
    'data',
    train=True,
    download=True,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]))
  
  #### Added Distributed Dataloader ####
  train_sampler = DistributedSampler(dataset=train_dataset)
  data_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler)
  ######################################
  
  model = Net().to(local_rank)
  #### Added Distributed Model ####
  ddp_model = DDP(model, device_ids=[local_rank], output_device=local_rank)
  #################################

  optimizer = optim.SGD(ddp_model.parameters(), lr=learning_rate, momentum=momentum)
  for epoch in range(1, num_epochs + 1):
    train_one_epoch(ddp_model, local_rank, data_loader, optimizer, epoch)
    
    if global_rank == 0: 
      save_checkpoint(directory, ddp_model, optimizer, epoch)
  
  # save out the model for test
  if global_rank == 0:
    mlflow.pytorch.log_model(ddp_model, "model")
    
    ddp_model.eval()
    test_dataset = datasets.MNIST(
      'data', 
      train=False,
      download=True,
      transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]))
    data_loader = torch.utils.data.DataLoader(test_dataset)    

    test_loss = 0
    for data, target in data_loader:
      device = torch.device('cuda')
      data, target = data.to(device), target.to(device)
      output = ddp_model(data)
      test_loss += F.nll_loss(output, target)
          
    test_loss /= len(data_loader.dataset)
    print("Average test loss: {}".format(test_loss.item()))
    
    mlflow.log_metric('test_loss', test_loss.item())

    
  dist.destroy_process_group()
  
  return "finished" # can return any picklable object


### Test without TorchDistributor

The following validates our training loop by running training on a single GPU.

In [0]:
# single node distributed run to quickly test that the whole process is working
with mlflow.start_run(run_name='single_node_distributed'):
  mlflow.log_param('run_type', 'test_dist_code')
  main_fn(single_node_single_gpu_dir)

### Single node multi-GPU training

PyTorch provides a [roundabout way](https://pytorch.org/tutorials/beginner/ddp_series_multigpu.html) for doing single node multi-GPU training. Databricks provides a more streamlined solution that allows you to move from single node multi-GPU to multi node training seamlessly. To do single node multi-GPU training on Databricks, you need to invoke the `TorchDistributor` API and set `num_processes` equal to the number of available GPUs on the driver node that you want to use and set `local_mode=True`.

In [0]:
single_node_multi_gpu_dir = create_log_dir()
print("Data is located at: ", single_node_multi_gpu_dir)

from pyspark.ml.torch.distributor import TorchDistributor

output = TorchDistributor(num_processes=2, local_mode=True, use_gpu=True).run(main_fn, single_node_multi_gpu_dir)
test(single_node_multi_gpu_dir)

### Multi-node training

To move from single node multi-GPU training to multi-node training, you just change `num_processes` to the number of GPUs that you want to use across all worker nodes. This example uses all available GPUs (`NUM_GPUS_PER_NODE * NUM_WORKERS`). You also change `local_mode` to `False`. Additionally, to configure how many GPUs to use for each Spark task that runs the train function, `set spark.task.resource.gpu.amount <num_gpus_per_task>` in the Spark Config cell on the cluster page before creating the cluster.

In [0]:
multi_node_dir = create_log_dir()
print("Data is located at: ", multi_node_dir)

output_dist = TorchDistributor(num_processes=2, local_mode=False, use_gpu=True).run(main_fn, multi_node_dir)
# test(multi_node_dir) # UNCOMMENT TO TEST THE MODEL (our driver node doesn't have a GPU)